# Airline AI Assistant with Tool Calling

Build a customer-support chatbot that can look up and update ticket prices. The notebook demonstrates how an LLM requests tools, how Python executes them, and how tool results are returned to the model.

## 1. Setup

Install the dependencies:

```bash
pip install openai gradio python-dotenv
```

Create a `.env` file with your API key and, optionally, a model name:

```text
OPENAI_API_KEY=your_api_key
OPENAI_MODEL=gpt-4.1-mini
```

In [ ]:
import json
import os
import sqlite3

import gradio as gr
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI()
MODEL = os.getenv("OPENAI_MODEL", "gpt-4.1-mini")

SYSTEM_MESSAGE = (
    "You are a helpful customer-support assistant for FlightAI. "
    "Give short, courteous, accurate answers. Use the available tools "
    "when customers ask about ticket prices or want to update them. "
    "If information is unavailable, say so."
)


## 2. Create a ticket-price database

SQLite stores ticket prices so the assistant can retrieve and update them.

In [ ]:
DB_PATH = "prices.db"

def initialise_database():
    with sqlite3.connect(DB_PATH) as connection:
        connection.execute(
            """
            CREATE TABLE IF NOT EXISTS prices (
                city TEXT PRIMARY KEY,
                price REAL NOT NULL
            )
            """
        )

def set_ticket_price(city, price):
    with sqlite3.connect(DB_PATH) as connection:
        connection.execute(
            """
            INSERT INTO prices (city, price)
            VALUES (?, ?)
            ON CONFLICT(city) DO UPDATE SET price = excluded.price
            """,
            (city.strip().lower(), float(price)),
        )
    return f"Ticket price for {city.title()} set to ${float(price):,.2f}."

def get_ticket_price(city):
    with sqlite3.connect(DB_PATH) as connection:
        row = connection.execute(
            "SELECT price FROM prices WHERE city = ?",
            (city.strip().lower(),),
        ).fetchone()

    if row is None:
        return f"No ticket price is currently available for {city.title()}."
    return f"The ticket price to {city.title()} is ${row[0]:,.2f}."

initialise_database()

# Example data for the chatbot
initial_prices = {
    "London": 799,
    "Paris": 899,
    "Tokyo": 1420,
    "Sydney": 2999,
}

for city, price in initial_prices.items():
    set_ticket_price(city, price)


## 3. Define the available tools

The tool schemas tell the model which functions it can request and what arguments each function requires. The model requests a tool; your Python application is responsible for executing it.

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_ticket_price",
            "description": "Look up the ticket price for a destination city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "Destination city",
                    }
                },
                "required": ["city"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "set_ticket_price",
            "description": "Set or update the ticket price for a destination city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "Destination city",
                    },
                    "price": {
                        "type": "number",
                        "description": "Ticket price in dollars",
                    },
                },
                "required": ["city", "price"],
                "additionalProperties": False,
            },
        },
    },
]


## 4. Execute tool calls

When the model requests a tool, inspect its name and arguments, run the matching Python function, and return the result with the tool-call ID. The ID links the result to the request.

In [ ]:
def execute_tool(tool_call):
    name = tool_call.function.name
    arguments = json.loads(tool_call.function.arguments)

    if name == "get_ticket_price":
        result = get_ticket_price(arguments["city"])
    elif name == "set_ticket_price":
        result = set_ticket_price(arguments["city"], arguments["price"])
    else:
        result = f"Unsupported tool: {name}"

    return {
        "role": "tool",
        "tool_call_id": tool_call.id,
        "content": result,
    }


## 5. Build the chatbot

The model may request one or more tools. The loop executes each requested tool, adds the results to the conversation, and asks the model to continue until it produces a normal response.

In [ ]:
def chat(message, history):
    messages = [
        {"role": "system", "content": SYSTEM_MESSAGE},
        *history,
        {"role": "user", "content": message},
    ]

    while True:
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
        )

        assistant_message = response.choices[0].message
        messages.append(assistant_message.model_dump(exclude_none=True))

        if not assistant_message.tool_calls:
            return assistant_message.content or ""

        for tool_call in assistant_message.tool_calls:
            messages.append(execute_tool(tool_call))


## 6. Launch the Gradio interface

Try asking for a ticket price, requesting a price update, and then checking the updated price.

In [ ]:
gr.ChatInterface(
    fn=chat,
    type="messages",
    title="FlightAI Airline Assistant",
    description="Ask about ticket prices or request a price update.",
).launch()


## Key concepts

- **Tool calling:** The model can request that your application run a named function.
- **Tool schema:** Describes each function and its expected arguments.
- **Tool execution:** Python validates the requested function name and runs the corresponding code.
- **Tool response:** The function result is sent back to the model using the matching tool-call ID.
- **Multiple tool calls:** A response can request several tools; the application processes each one.
- **Tool loop:** The model can request additional tools before returning its final answer.
- **Database integration:** SQLite provides persistent storage for ticket prices.

## Business application

Tool calling connects an LLM to external actions and data. In a real airline application, similar tools could query flight availability, retrieve booking details, or interact with approved booking and payment services. Protect write operations with authentication, validation, and appropriate user confirmation.